In [1]:
%reload_ext autotime
import pandas as pd
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
from pprint import pprint
import json5 as json # This is a more forgiving JSON parser that can handle comments, single quotes, and trailing commas
from glob import glob
from tqdm import tqdm
from openai import OpenAI
import base64
files = sorted(glob("../supplements_videos/*.json"))
print(len(files))

2330
time: 981 ms (started: 2026-06-05 12:46:47 +12:00)


In [2]:
print(pd.to_timedelta("00:" + pd.read_csv("../data/supplements.csv").duration).describe())
print("Total video time:", pd.to_timedelta("00:" + pd.read_csv("../data/supplements.csv").duration).sum())

count                      5690
mean     0 days 00:00:58.261335
std      0 days 00:00:39.947171
min             0 days 00:00:02
25%             0 days 00:00:27
50%             0 days 00:00:53
75%             0 days 00:01:21
max             0 days 00:03:00
Name: duration, dtype: object
Total video time: 3 days 20:05:07
time: 70.1 ms (started: 2026-06-05 12:46:48 +12:00)


In [3]:
json_filename = files[2]
with open(json_filename) as f:
    data = json.load(f)
display(pd.json_normalize(data))
print(data["description"])

,id,formats,channel,channel_id,uploader,uploader_id,channel_url,uploader_url,track,artists,duration,title,description,timestamp,view_count,like_count,repost_count,comment_count,save_count,thumbnails,webpage_url,webpage_url_basename,webpage_url_domain,extractor,extractor_key,thumbnail,display_id,fulltitle,duration_string,upload_date,artist,epoch,ext,vcodec,acodec,format_id,tbr,quality,preference,filesize,width,height,url,protocol,video_ext,audio_ext,resolution,dynamic_range,aspect_ratio,cookies,format,_type,http_headers.User-Agent,http_headers.Accept,http_headers.Accept-Language,http_headers.Sec-Fetch-Mode,http_headers.Referer,_version.version,_version.release_git_head,_version.repository
0,7638997182208544030,"[{'ext': 'mp4', 'vcodec': 'h264', 'acodec': 'aac', 'format_id': 'h264_540p_423454-0', 'tbr': 423...",Mi vida 😍,MS4wLjABAAAAljTo2gcSK_djf6GSIXY_EomH8KxCNMN-m7XLw4ZbtpUnL-cmTrRtYRfVxLxPkiYz,lomejordemivida7,6697254127905686534,https://www.tiktok.com/@MS4wLjABAAAAljTo2gcSK_djf6GSIXY_EomH8KxCNMN-m7XLw4ZbtpUnL-cmTrRtYRfVxLxP...,https://www.tiktok.com/@lomejordemivida7,sonido original,[Mi vida 😍],19,#BigDreamsStartSmall #starterpicks #menopausesupport #hormonesupport,#BigDreamsStartSmall #starterpicks #menopausesupport #hormonesupport,1778592668,1625,11,0,0,1,"[{'id': 'dynamicCover', 'url': 'https://p16-common-sign.tiktokcdn.com/tos-useast8-p-0068-tx2/oUn...",https://www.tiktok.com/@lomejordemivida7/video/7638997182208544030,7638997182208544030,tiktok.com,TikTok,TikTok,https://p16-common-sign.tiktokcdn.com/tos-useast8-p-0068-tx2/ooIQICOIILAGwBCAQjmGq1volKVVaaUGfIf...,7638997182208544030,#BigDreamsStartSmall #starterpicks #menopausesupport #hormonesupport,19,20260512,Mi vida 😍,1780619335,mp4,h265,aac,bytevc1_1080p_1361165-1,1361,3,-1,3270030,1080,1920,https://v19-webapp-prime.tiktok.com/video/tos/maliva/tos-maliva-ve-0068c799-us/oMREDAi9qRRM2rBAi...,https,mp4,none,1080x1920,SDR,0.56,"msToken=""0l5NrxqxQknMMk_1ZjxdexqBHbrRvY3LCBjxcqacdS74hkmJtkQ2U73skG3MKitiEA2rJXTBrDFxY0y3_FZ9n5j...",bytevc1_1080p_1361165-1 - 1080x1920,video,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0....","text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8","en-us,en;q=0.5",navigate,https://www.tiktok.com/@lomejordemivida7/video/7638997182208544030,2026.03.17,04d6974f502bbdfaed72c624344f262e30ad9708,yt-dlp/yt-dlp


#BigDreamsStartSmall #starterpicks #menopausesupport #hormonesupport 
time: 301 ms (started: 2026-06-05 12:46:48 +12:00)


In [4]:
def get_prompt(data):
    return f"""This is a video downloaded from {data['extractor']}. Here's the description of the video: {data['description']}.

        The creator of the video is {data.get('channel', 'an unknown channel')} ({data.get('uploader', 'an unknown uploader')})
        It has {data.get('like_count', 'an unknown number of')} likes, {data.get('view_count', 'an unknown number of')} views, and {data.get('comment_count', 'an unknown number of')} comments.
        Taking into account this description, and the video, extract the following information, in JSON format:
        description: What is happening in the video? Provide a detailed description of the actions, context, and any notable elements present in the video.
        transcript: If there is any spoken content in the video, transcribe it into English. If there is no spoken content, indicate "No spoken content".
        tone: What is the overall tone or mood of the video? Is it humorous, serious, educational, emotional, etc.?
        supplements: Does the video mention any supplements, vitamins, or medications? If so, list them. If not, indicate "No supplements mentioned".
        active_ingredients: If any supplements are mentioned, list the active ingredients in those supplements. If no supplements are mentioned, indicate "No active ingredients mentioned".
        symptoms: Does the video mention any specific symptoms, conditions, or health issues? If so, list them. If not, indicate "No symptoms mentioned".
        menopause: Is the video specifically targeting the supplement towards menopause-related symptoms or conditions? Answer True or False.
        language: What language is this video in?
        marketing: Is this video promoting or advertising any product, service, brand, or organization? If so, what is it? Otherwise, indicate "No marketing content".
        job: For the main speaker, what is their job or profession? If it is not mentioned in the video, indicate "No job information". A comma separated string, one or more of the following: therapist, psychologist, pediatrician, doctor, nurse, teacher, professor, social worker, counselor, coach, influencer, content creator?
        sentiment: Does this video recommend a particular supplement, discourage it, or is it neutral? One of negative, neutral or positive
        criticism: If the video is critical of a particular supplement, what are the main criticisms mentioned? 
        alternative_strategies: Does the video mention any alternative strategies to supplements? If so, what are they? A comma separated string. If no alternatives are mentioned, indicate "No alternative strategies mentioned".
        usefulness: Rate the overall usefulness of the video on a scale from 1 to 10, where 1 is not useful at all and 10 is extremely useful.
        misleading: Rate the extent to which the video contains misleading or inaccurate information on a scale from 1 to 10, where 1 is not misleading at all and 10 is extremely misleading.
        quality: Rate the overall quality of the video on a scale from 1 to 10, where 1 is very poor quality and 10 is excellent quality.
        personal_experience: Does the speaker mention any personal experience with supplements? If so, briefly summarize it.

        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid
    """
print(get_prompt(data))

This is a video downloaded from TikTok. Here's the description of the video: #BigDreamsStartSmall #starterpicks #menopausesupport #hormonesupport .

        The creator of the video is Mi vida 😍 (lomejordemivida7)
        It has 11 likes, 1625 views, and 0 comments.
        Taking into account this description, and the video, extract the following information, in JSON format:
        description: What is happening in the video? Provide a detailed description of the actions, context, and any notable elements present in the video.
        transcript: If there is any spoken content in the video, transcribe it into English. If there is no spoken content, indicate "No spoken content".
        tone: What is the overall tone or mood of the video? Is it humorous, serious, educational, emotional, etc.?
        supplements: Does the video mention any supplements, vitamins, or medications? If so, list them. If not, indicate "No supplements mentioned".
        active_ingredients: If any supplement

In [ ]:
client = OpenAI(base_url="https://ai.cer-sandbox.cloud.edu.au/v1", api_key="not needed")

video_filename = json_filename.replace("info.json", data["ext"])
with open(video_filename, "rb") as video_file:
    base64_video = "data:video/" + data["ext"] + ";base64," + base64.b64encode(video_file.read()).decode("utf-8")
print(len(base64_video))
 
response = client.chat.completions.create(
    model="nemotron_3_nano_omni",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": base64_video}},
                {"type": "text", "text": get_prompt(data)},
            ],
        }
    ],
    max_tokens=20480,
    temperature=0.6,
    top_p=0.95,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False,
        },
        "mm_processor_kwargs": {"use_audio_in_video": True},
    },
)
print(response.choices[0].message.content)

4360062

{
  "description": "A woman with glasses and her hair in a bun is holding a white package with a pink circle and text on it. She opens the package, revealing pink circular patches inside a clear plastic sleeve. She takes out one patch and holds it up to her arm, showing it to the camera. She then holds the package up again, displaying the front and back of the packaging.",
  "transcript": "This is in the first menopause, a post-menopause. These little patches are very good because they minimize all those symptoms that we have in all those periods. And the price of this is very good. I bought them to check the price and the information, they are 30, for a month and it is very good.",
  "tone": "educational",
  "supplements": "Menopause patches",
  "active_ingredients": "No active ingredients mentioned",
  "symptoms": "No symptoms mentioned",
  "menopause": true,
  "language": "Spanish",
  "marketing": "Menopause patches",
  "job": "influencer",
  "sentiment": "positive",
  "cri